<a href="https://colab.research.google.com/github/Giocrisrai/taller-ia-agentes-biblioteca/blob/main/notebooks/02_Taller_Practico_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

> **Ábrelo en Colab con el botón de arriba.** Una vez dentro, menú `Archivo` →
> `Guardar una copia en Drive`, para tener tu propia versión editable.


# Taller Práctico 2 — Varias herramientas, memoria y límites

**DUOC UC · Bibliotecas** · Curso de IA Aplicando Agentes · Sesión práctica 2 de 2

---

El viernes construiste un agente con **una** herramienta. Hoy le vas a dar **varias**,
y el agente tendrá que **decidir cuál usar** en cada caso.

### Repaso relámpago

| Pieza | Qué es |
|---|---|
| **Herramienta** | Una función de Python con `@tool`. El modelo la elige por su **descripción** |
| **Modelo** | El que razona y decide. Nunca ejecuta código |
| **AgentExecutor** | El motor que sí ejecuta la herramienta y devuelve el resultado al modelo |
| **Ciclo ReAct** | Pensamiento → Acción → Observación, repetido hasta responder |

### El camino de hoy: 10 pasos

| | Paso | Qué haces |
|---|---|---|
| 🔑 | **1** | Preparas el entorno y la llave |
| 🔧 | **2** | Creas tres herramientas de biblioteca |
| 🤖 | **3** | El agente elige una de las tres |
| 🤖 | **4** | El agente encadena dos en una sola consulta |
| 🧠 | **5** | Compruebas que el agente **no** recuerda nada |
| 🧠 | **6** | Le das memoria |
| 🧠 | **7** | Miras la memoria por dentro |
| 🛡️ | **8** | Le pones límites y los ves actuar |
| ✏️ | **9** | Agregas tu cuarta herramienta |
| 💾 | **10** | Guardas tu copia y recibes la tarea |

> 🙋 **Si algo falla, levanta la mano.**


---
---

# 🔑 Paso 1 · Prepara el entorno

⏱️ *5 minutos* · Son tres celdas seguidas. Ejecútalas en orden.

Es lo mismo del viernes. Si ya tienes tu llave guardada en los Secrets de Colab, no hay
nada que hacer: solo ejecutar.

### Si el viernes no lograste la llave

Hazlo ahora, son 3 minutos:

**1.a** Entra a **[console.groq.com](https://console.groq.com/)** → crea tu cuenta →
menú izquierdo **API Keys** → **Create API Key** → cópiala (empieza por `gsk_`).

**1.b** Aquí en Colab: barra lateral izquierda → ícono de **llave** → **Add new secret** →
en **Name** escribe exactamente `GROQ_API_KEY` → en **Value** pega tu llave.

**1.c** ⚠️ **Activa el interruptor** *Notebook access*. Sin eso el notebook no la ve.

> ⚠️ **Hoy vas a necesitar la llave de verdad.** El modo simulado te deja seguir la clase,
> pero no muestra al agente razonando, y hoy eso es justo lo que hay que ver.

> 🤔 **«¿Por qué dice `openai` si estamos usando Groq?»** Es la pregunta que siempre
> sale, y la respuesta es que **no estás usando OpenAI en ningún momento**:
>
> | Dónde lo ves | Qué es de verdad |
> |---|---|
> | `openai/gpt-oss-120b` | El **nombre del modelo**. GPT-OSS son unos pesos abiertos que OpenAI publicó, y **Groq los hospeda y ejecuta**. Se cobra a tu cupo de Groq. |
> | `create_openai_tools_agent` | El **nombre de una función de LangChain**, por el formato de llamada a herramientas que popularizó OpenAI y que Groq también implementa. |
>
> No necesitas cuenta de OpenAI ni `OPENAI_API_KEY`. **Todo el taller funciona con tu
> única llave de Groq.**

> ⚠️ **La primera vez, Colab te va a mostrar un aviso.** Dice
> *«Warning: This notebook was not authored by Google»* y aparece porque el notebook viene
> de GitHub. **Es normal.** Haz clic en **`Run anyway`** (Ejecutar de todos modos) y sigue.


In [ ]:
# ▶ PASO 1a · Instala las librerías. Unos 30 segundos.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain langchain-classic langchain-groq groq
    print("✅ Librerías instaladas.")
else:
    print("✅ Entorno local detectado.")


In [ ]:
# ▶ PASO 1b · Carga tu llave.
import os

try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

# ⚠️ Hoy usamos el modelo RÁPIDO, no el grande.
# Motivo: cuando un agente encadena varias herramientas seguidas, gpt-oss-20b
# se equivoca menos con el formato de las llamadas que el de 120b. Está medido.
MODELO = "openai/gpt-oss-20b"

MODO_SIMULADO = not os.getenv("GROQ_API_KEY")

if MODO_SIMULADO:
    print("⚠️  Sin llave. Revisa el nombre del Secret (GROQ_API_KEY) y el")
    print("    interruptor 'Notebook access'. Ver los puntos 1.a a 1.c de arriba.")
    print("    Sigues en MODO SIMULADO.")
else:
    print(f"✅ Paso 1 listo. Llave cargada. Modelo de hoy: {MODELO}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ▶ PASO 1c · Celda técnica: es lo mismo que armamos el viernes, reunido aquí.
# No necesitas leerla. Ejecútala y sigue al Paso 2.
# ─────────────────────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent


class AgenteSimulado:
    """Agente de emergencia que funciona SIN modelo de lenguaje.

    No razona: elige la herramienta comparando palabras de la pregunta con el
    nombre y la descripción de cada una, y adivina los parámetros con reglas
    fijas. Es aproximado a propósito. Sirve para que puedas seguir la clase si
    no lograste tu llave, no para aprender cómo decide un agente de verdad.
    """

    import re as _re

    DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

    def __init__(self, herramientas):
        self.herramientas = herramientas

    def _campos(self, herramienta):
        """Nombres de los parámetros que la herramienta necesita."""
        try:
            return list(herramienta.args.keys())
        except Exception:
            return []

    def _adivinar(self, campo, pregunta):
        """Intenta sacar el valor de un parámetro con reglas simples."""
        c = campo.lower()
        if "hora" in c:
            m = self._re.search(r"\b(\d{1,2}:\d{2})\b", pregunta)
            return m.group(1) if m else None
        if "sala" in c:
            m = self._re.search(r"[Ss]ala\s*(\d+)", pregunta)
            return f"Sala {m.group(1)}" if m else None
        if "codigo" in c or "código" in c:
            m = self._re.search(r"\b([A-Za-z]-?\d{3})\b", pregunta)
            return m.group(1).upper() if m else None
        if "dia" in c or "día" in c:
            for d in self.DIAS:
                if d in pregunta.lower():
                    return d
            return None
        if "titulo" in c or "título" in c or "libro" in c:
            m = self._re.search(r"libro\s+(?:llamado\s+|titulado\s+)?([A-ZÁÉÍÓÚÑ]\w*(?:\s+[A-ZÁÉÍÓÚÑ]\w*)*)", pregunta)
            return m.group(1) if m else None
        return None

    def invoke(self, entrada):
        pregunta = entrada.get("input", "")
        historial = entrada.get("chat_history", [])
        palabras = set(pregunta.lower().replace("¿", " ").replace("?", " ").split())

        mejor, top = None, 0
        for h in self.herramientas:
            texto = (h.name + " " + (h.description or "")).lower()
            p = sum(1 for w in palabras if len(w) > 3 and w in texto)
            if p > top:
                mejor, top = h, p

        print("> Entrando en la cadena del agente...  [MODO SIMULADO]")
        if historial:
            print(f"  (recibe {len(historial)} mensajes de lo ya conversado)")

        if mejor is None:
            print("Pensamiento: ninguna herramienta encaja con la pregunta.")
            salida = "(modo simulado) No tengo una herramienta para responder eso."
            print("> Cadena terminada.")
            return {"output": salida}

        print(f"Pensamiento: esto parece requerir la herramienta '{mejor.name}'.")

        campos = self._campos(mejor)
        argumentos, faltan = {}, []
        for campo in campos:
            valor = self._adivinar(campo, pregunta)
            if valor is None:
                faltan.append(campo)
            else:
                argumentos[campo] = valor

        if faltan:
            print(f"Acción: {mejor.name}  ← no pude deducir: {', '.join(faltan)}")
            print("   (el modo simulado adivina los parámetros con reglas fijas.")
            print("    Un agente real los deduce leyendo la frase completa.)")
            salida = f"(modo simulado) Necesitaría el valor de: {', '.join(faltan)}."
            print("> Cadena terminada.")
            return {"output": salida}

        print(f"Acción: {mejor.name}({argumentos})")
        try:
            observacion = mejor.invoke(argumentos)
        except Exception as e:
            observacion = f"(no pude ejecutarla: {type(e).__name__})"
        print(f"Observación: {observacion}")
        print("> Cadena terminada.")
        return {"output": f"(modo simulado) {observacion}"}


INSTRUCCIONES_POR_DEFECTO = (
    "Eres un asistente de la biblioteca de DUOC UC. "
    "Responde en español, breve y amable. "
    "Si tienes una herramienta que puede darte el dato, úsala en vez de suponerlo."
)


def armar_agente(herramientas, instrucciones=INSTRUCCIONES_POR_DEFECTO, max_pasos=6):
    """Une modelo + herramientas + instrucciones y devuelve un agente listo."""
    if MODO_SIMULADO:
        return AgenteSimulado(herramientas)
    prompt = ChatPromptTemplate.from_messages([
        ("system", instrucciones),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])
    llm = ChatGroq(model=MODELO, temperature=0)
    cerebro = create_openai_tools_agent(llm, herramientas, prompt)
    return AgentExecutor(
        agent=cerebro, tools=herramientas, verbose=True,
        max_iterations=max_pasos, handle_parsing_errors=True,
    )


def preguntar(agente, mensaje, historial=None):
    """Envía un mensaje al agente y devuelve su respuesta. Nunca lanza excepción."""
    try:
        r = agente.invoke({"input": mensaje, "chat_history": historial or []})
        return r["output"]
    except Exception as e:
        return f"❌ {type(e).__name__}: {e}\n   Si dice 'tool_use_failed', vuelve a ejecutar la celda."


print("✅ Paso 1 completo. Sigue con el Paso 2.")


---
---

# 🔧 Paso 2 · Crea tres herramientas de biblioteca

⏱️ *5 minutos*

Vamos a darle al agente **tres** herramientas. Fíjate en la última columna, porque marca
una diferencia que importa:

| Herramienta | Qué hace | ¿De dónde salen los datos? |
|---|---|---|
| `buscar_libro` | Busca un libro en el catálogo | 🌍 **API real** de Open Library, 40 millones de libros |
| `salas_disponibles` | Dice qué salas quedan libres | 🏫 Simulada: DUOC no tiene una API pública de salas |
| `reservar_sala` | Reserva una sala **de verdad** | 🏫 Simulada, y menos mal — **cambia el estado del mundo** |

### Por qué dos están simuladas, y por qué está bien

`buscar_libro` habla con un catálogo real porque **existe** una API pública para eso.
Las salas de DUOC no la tienen: en un sistema real esas dos funciones consultarían la base
de datos interna de la biblioteca. Lo que cambia es **el contenido de la función**, no el
agente: el agente no sabe ni le importa de dónde vienen los datos.

Y hay una segunda razón, más práctica: `reservar_sala` **modifica cosas**. Simularla es lo
que nos permite provocar el fallo a propósito en el Paso 8 sin reservarle una sala de
verdad a nadie.

> 💡 **La idea que se llevan:** una herramienta es una función de Python. Puede leer un
> diccionario, llamar a una API, consultar una base de datos o mandar un correo. El agente
> se comporta igual en los cuatro casos.


In [ ]:
# ▶ PASO 2 · Los datos y las tres herramientas.
import json
import urllib.parse
import urllib.request
from langchain_core.tools import tool

# ── Datos internos simulados (las salas de DUOC no tienen API pública) ────────
SALAS = {
    "Sala 204": ["10:00", "11:00", "16:00"],
    "Sala 301": ["09:00", "16:00", "18:00"],
}

RESERVAS = []   # aquí se van guardando las reservas hechas


# ── Herramienta 1: catálogo REAL ─────────────────────────────────────────────

def _consultar_catalogo(titulo, intentos=2):
    """Pregunta a Open Library. Reintenta una vez si la red falla."""
    url = "https://openlibrary.org/search.json?" + urllib.parse.urlencode({
        "q": titulo, "limit": 3,
        "fields": "title,author_name,first_publish_year,edition_count",
    })
    for intento in range(intentos):
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "Taller-DUOC-Bibliotecas/1.0"})
            with urllib.request.urlopen(pedido, timeout=15) as respuesta:
                return json.loads(respuesta.read()), None
        except Exception as e:
            error = type(e).__name__
    return None, error


@tool
def buscar_libro(titulo: str) -> str:
    """Busca un libro en el catálogo mundial de Open Library y devuelve su autor, el año
    de publicación y cuántas ediciones existen. Úsala siempre que pregunten por un libro
    concreto, por su título."""
    datos, error = _consultar_catalogo(titulo)
    if error:
        return f"No pude consultar el catálogo ahora ({error}). Vuelve a intentarlo."
    if not datos.get("docs"):
        return f"No encontré '{titulo}' en el catálogo."
    resultados = []
    for d in datos["docs"][:3]:
        autor = (d.get("author_name") or ["autor desconocido"])[0]
        anio = d.get("first_publish_year", "año desconocido")
        resultados.append(f"«{d['title']}» de {autor}, {anio} ({d.get('edition_count', 0)} ediciones)")
    return f"Encontré {datos['numFound']} resultados. Los primeros: " + " | ".join(resultados)


# ── Herramientas 2 y 3: sistema interno, simulado ────────────────────────────

@tool
def salas_disponibles(hora: str) -> str:
    """Indica qué salas de estudio están libres a una hora determinada.
    Recibe la hora en formato HH:MM, por ejemplo '16:00'."""
    hora = hora.strip()
    libres = [s for s, horas in SALAS.items() if hora in horas and (s, hora) not in RESERVAS]
    if not libres:
        return f"No hay salas libres a las {hora}."
    return f"A las {hora} están libres: {', '.join(libres)}."


@tool
def reservar_sala(sala: str, hora: str) -> str:
    """Reserva una sala de estudio a una hora concreta. Recibe el nombre de la sala
    (por ejemplo 'Sala 204') y la hora en formato HH:MM. Solo debe usarse cuando la
    persona ya confirmó explícitamente que quiere reservar."""
    sala, hora = sala.strip(), hora.strip()
    if sala not in SALAS:
        return f"'{sala}' no existe. Las salas son: {', '.join(SALAS)}."
    if hora not in SALAS[sala]:
        return f"La {sala} no tiene disponibilidad a las {hora}."
    if (sala, hora) in RESERVAS:
        return f"La {sala} ya estaba reservada a las {hora}."
    RESERVAS.append((sala, hora))
    return f"✅ {sala} reservada a las {hora}. Código de reserva: R-{len(RESERVAS):03d}."


herramientas = [buscar_libro, salas_disponibles, reservar_sala]

print("✅ Paso 2 listo. 3 herramientas creadas:")
print("   · buscar_libro       🌍 catálogo real (Open Library)")
print("   · salas_disponibles  🏫 sistema interno simulado")
print("   · reservar_sala      🏫 sistema interno simulado")
print()
print("Prueba directa del catálogo real:")
print(" ", buscar_libro.invoke({"titulo": "El principito"}))


---
---

# 🤖 Paso 3 · El agente elige una de las tres

⏱️ *5 minutos*

Antes de ejecutar, **piensa un momento**: le vamos a preguntar *"¿Tienen el libro Sapiens?"*.

- ¿Cuál de las tres herramientas va a usar?
- ¿Cómo lo sabe, si nadie se lo dijo?

Ejecuta y comprueba tu respuesta mirando la línea `Invoking:` de la traza.


In [ ]:
# ▶ PASO 3 · Una pregunta que solo necesita UNA de las tres herramientas.

agente = armar_agente(herramientas)

print(preguntar(agente, "¿Tienen el libro Sapiens?"))


### 🔑 ¿Cómo supo cuál usar?

Por **la descripción**. Es lo único que el modelo ve de cada herramienta:
no lee el código, no hay reglas ni condicionales escritos por nadie.

Por eso una herramienta mal descrita es una herramienta que el agente **no va a usar nunca**.


---

# 🤖 Paso 4 · El agente encadena dos herramientas

⏱️ *8 minutos* · **Este es el momento central de la sesión.**

Ahora una consulta que necesita **dos** herramientas seguidas:

> *"Necesito el libro Sapiens y además una sala para estudiar a las 16:00."*

Ejecuta y **cuenta cuántas veces aparece `Invoking:` en la traza**.


In [ ]:
# ▶ PASO 4 · Una consulta que necesita DOS herramientas seguidas.

print(preguntar(agente, "Necesito el libro Sapiens y además una sala para estudiar a las 16:00."))


### 🤔 ¿Qué pasó aquí?

En la traza deberías haber visto **dos** bloques `Invoking:`, uno detrás de otro.

El agente:

1. Leyó tu consulta y se dio cuenta de que tenía **dos partes**
2. Llamó a `buscar_libro` y recibió la observación
3. **Con ese resultado ya en mano**, decidió que aún faltaba y llamó a `salas_disponibles`
4. Solo entonces redactó la respuesta final juntando ambos datos

Nadie le dijo "primero busca el libro y luego la sala". **Eso lo decidió él.**
Eso es la **orquestación** de la que hablamos en la teoría.

> ⚠️ Si el agente se saltó una herramienta o inventó un dato, no está roto: es
> exactamente el riesgo de **alucinación** del que hablamos. Vuelve a ejecutar la celda
> y compara: **no siempre hace lo mismo.**


---
---

# 🧠 Paso 5 · Comprueba que el agente no recuerda nada

⏱️ *4 minutos*

Hasta ahora cada consulta empezaba de cero. Vamos a comprobarlo.

En la celda de abajo hay **dos turnos seguidos**: en el primero te presentas, y en el
segundo preguntas tu propio nombre.


In [ ]:
# ▶ PASO 5 · Dos mensajes seguidos, SIN memoria.

print("TURNO 1")
print(preguntar(agente, "Hola, me llamo Camila y busco el libro Sapiens."))

print()
print("TURNO 2")
print(preguntar(agente, "¿Cómo me llamo?"))


### El agente no supo tu nombre

Aunque se lo acababas de decir. **¿Por qué?**

Porque cada llamada a `invoke` es **independiente**: lo único que recibe el modelo es el
mensaje de ese turno. No hay ningún sitio donde se guarde lo anterior.

> **La memoria de un agente no es magia: es una lista de mensajes que le vuelves a mandar
> en cada turno.** Nada más que eso. Es lo que harás en el Paso 6.


---

# 🧠 Paso 6 · Dale memoria

⏱️ *6 minutos*

Fíjate en la función `conversar()` de la celda: hace tres cosas, y ninguna es complicada.

1. Le pasa al agente **el historial completo** junto con el mensaje nuevo
2. Guarda en el historial **lo que dijo la persona**
3. Guarda en el historial **lo que respondió el agente**

Los mismos dos turnos del Paso 5. Ahora sí debería saber tu nombre.


In [ ]:
# ▶ PASO 6 · Los mismos dos turnos, ahora CON memoria.

historial = []   # <- aquí vive la memoria: una lista de mensajes


def conversar(agente, mensaje, historial):
    """Envía un mensaje incluyendo todo lo conversado antes, y guarda el nuevo turno."""
    respuesta = preguntar(agente, mensaje, historial)      # (1) le pasamos lo anterior
    historial.append(HumanMessage(content=mensaje))        # (2) lo que dijo la persona
    historial.append(AIMessage(content=respuesta))         # (3) lo que respondió el agente
    return respuesta


print("TURNO 1")
print(conversar(agente, "Hola, me llamo Camila y busco el libro Sapiens.", historial))

print()
print("TURNO 2")
print(conversar(agente, "¿Cómo me llamo?", historial))


---

# 🧠 Paso 7 · Mira la memoria por dentro

⏱️ *3 minutos*

La memoria no es una caja negra. Ejecuta la celda y **mírala**: son cuatro mensajes
alternando persona y agente.


In [ ]:
# ▶ PASO 7 · La memoria por dentro. No es más que esto.

print(f"El historial tiene {len(historial)} mensajes:\n")
for i, m in enumerate(historial, 1):
    quien = "PERSONA" if isinstance(m, HumanMessage) else "AGENTE "
    print(f"{i}. [{quien}] {m.content[:90]}")


### 💡 La consecuencia práctica

Si la memoria es una lista que **se reenvía completa en cada turno**, entonces:

- Una conversación **larga cuesta más** — más texto enviado son más tokens, y más tokens es más dinero
- Y en algún momento **ya no cabe** en el modelo

Por eso los sistemas reales no guardan todo: resumen lo viejo, o guardan solo lo
relevante. Ese es el problema que resuelven las memorias avanzadas.

> 📌 En el PPT teórico esto aparecía como `ConversationBufferMemory`. Es la misma idea
> con otro nombre: una caja que acumula los mensajes. Aquí la escribimos a mano para que
> se vea que por dentro **es solo una lista**.


---
---

# 🛡️ Paso 8 · Ponle límites y míralos actuar

⏱️ *10 minutos*

`reservar_sala` es distinta a las otras dos: **cambia el estado del mundo**. Si el agente
la usa por su cuenta, alguien se queda con una sala reservada que no pidió.

Aquí se ven **tres tipos de límite** en el mismo agente. Los tres a la vez, porque
ninguno basta solo:

| | Límite | Dónde vive | Qué evita |
|---|---|---|---|
| **1** | Instrucción de confirmar | En el `system` prompt | Que actúe sin permiso |
| **2** | `max_iterations` | En el `AgentExecutor` | Que entre en bucle infinito y gaste tu cupo |
| **3** | Validación en la función | Dentro de la herramienta | Que un dato inventado cause daño |

El tercero es el más importante y el que más se olvida: **el modelo puede alucinar, tu
código no.** Por eso `reservar_sala` comprueba que la sala exista antes de reservar.

### 8.1 · Pide reservar sin elegir sala

El agente **no debería** reservar todavía. Mira la última línea: `RESERVAS` tiene que
seguir vacía.


In [ ]:
# ▶ PASO 8.1 · Le pedimos que reserve SIN haber elegido sala. No debería reservar.

INSTRUCCIONES_CON_LIMITES = (
    "Eres un asistente de la biblioteca de DUOC UC. Responde en español, breve y amable. "
    "Consulta libros y salas libremente. "
    "REGLA CRÍTICA: nunca uses la herramienta reservar_sala sin que la persona haya "
    "confirmado explícitamente que quiere reservar esa sala a esa hora. Si te piden "
    "reservar sin haber elegido sala, primero muestra las opciones y pregunta cuál quiere."
)

agente_cuidadoso = armar_agente(
    herramientas,
    instrucciones=INSTRUCCIONES_CON_LIMITES,
    max_pasos=4,      # <- límite 2: nunca más de 4 vueltas del ciclo
)

print(preguntar(agente_cuidadoso, "Resérvame una sala a las 16:00."))

print()
print("Reservas registradas hasta ahora:", RESERVAS or "ninguna")


### 8.2 · Ahora sí confirma

Comprueba que la reserva **sí** aparece al final.


In [ ]:
# ▶ PASO 8.2 · Ahora confirmamos explícitamente.

print(preguntar(agente_cuidadoso, "Sí, confirmo: reserva la Sala 204 a las 16:00."))

print()
print("Reservas registradas ahora:", RESERVAS or "ninguna")


### 🔑 La idea que se llevan de hoy

> ## El agente propone, tu código dispone.

El modelo decidió *qué* hacer. Pero quien decidió *qué es posible hacer* fuiste tú,
al escribir las herramientas y sus validaciones.

**Un agente nunca es más peligroso que las herramientas que le diste.**
Si no le das una herramienta para borrar, no puede borrar.


---
---

# ✏️ Paso 9 · Tu turno

⏱️ *8 minutos*

Agrega una **cuarta herramienta** a este agente.

### 9.1 · Ejecuta el ejemplo y pruébalo dos veces

`renovar_prestamo` ya funciona. Pruébalo con **dos códigos distintos**:

| Código | Qué pasa | Por qué |
|---|---|---|
| `B-045` | Se renueva | Tiene 0 renovaciones |
| `B-112` | **Lo rechaza** | Ya se renovó 2 veces |

Y fíjate en algo: **quien dijo que no no fue el modelo. Fue el `if` que escribiste tú.**

### 9.2 · Ahora escribe la tuya

- `cuantos_computadores_libres()`
- `buscar_por_autor(autor)`
- `multa_pendiente(rut)`
- lo que se te ocurra

**Dos reglas para que funcione bien:**

1. La **descripción entre comillas triples** debe decir *cuándo* usarla. Es lo único que
   lee el modelo
2. Si tu herramienta **cambia algo**, valida los datos dentro de la función


In [ ]:
# ▶ PASO 9 · Tu cuarta herramienta. Ejecútala tal cual y después cámbiala.

PRESTAMOS = {
    "B-045": {"libro": "Sapiens", "renovaciones": 0, "vence": "05-09-2026"},
    "B-112": {"libro": "Clean Code", "renovaciones": 2, "vence": "08-09-2026"},
}


@tool
def renovar_prestamo(codigo: str) -> str:
    """Renueva por 7 días más un préstamo activo de la biblioteca. Recibe el código
    del préstamo, por ejemplo 'B-045'. Un préstamo se puede renovar como máximo 2 veces."""
    codigo = codigo.strip().upper()
    if codigo not in PRESTAMOS:
        return f"No existe el préstamo {codigo}."
    prestamo = PRESTAMOS[codigo]
    if prestamo["renovaciones"] >= 2:            # <- la validación que el modelo no puede saltarse
        return f"El préstamo {codigo} ya se renovó 2 veces. Hay que devolver el libro."
    prestamo["renovaciones"] += 1
    return (f"✅ Préstamo {codigo} ('{prestamo['libro']}') renovado. "
            f"Renovaciones usadas: {prestamo['renovaciones']} de 2.")


# Se la damos al agente junto con las otras tres.
herramientas_v2 = [buscar_libro, salas_disponibles, reservar_sala, renovar_prestamo]
agente_v2 = armar_agente(herramientas_v2, instrucciones=INSTRUCCIONES_CON_LIMITES)

print("✅ Agente con 4 herramientas.")


In [ ]:
# ▶ PASO 9 · Pruébalo. Cambia la pregunta por la que tú quieras.

print(preguntar(agente_v2, "Quiero renovar mi préstamo B-045."))

# Después prueba con B-112: ese ya se renovó 2 veces y la herramienta lo rechaza.
# Fíjate en que quien dice que no es TU código, no el modelo.


---
---

# 💾 Paso 10 · Guarda tu copia y recibe la tarea

⏱️ *5 minutos*

### 10.1 · Guarda ahora

Menú de arriba → **`Archivo`** → **`Guardar una copia en Drive`**

Si no lo haces, pierdes todo lo de hoy.

---

## Lo que construiste en las dos sesiones

| Concepto | Qué es | Dónde lo viste |
|---|---|---|
| Herramienta | Función de Python que el agente puede invocar | Pasos 2 y 9 |
| Encadenamiento | Varias tools seguidas en una sola consulta | Paso 4 |
| Memoria | Lista de mensajes que se reenvía cada turno | Pasos 6 y 7 |
| Límite | Instrucción, tope de pasos o validación en el código | Paso 8 |
| Ciclo ReAct | Pensamiento → Acción → Observación | Toda la traza |

---

## 📋 10.2 · La tarea evaluada

**Construye tu propio agente**, en el dominio que tú quieras: una cafetería, un gimnasio,
la secretaría de tu carrera, un taller mecánico, tu propio emprendimiento.

Tienes la **plantilla lista** en `03_Tarea_Plantilla.ipynb`, con todo el andamiaje puesto
y los pasos numerados igual que aquí.

| # | Requisito | Puntos |
|---|---|---|
| 1 | **3 herramientas propias** con `@tool` y buena descripción | 20 |
| 2 | Una consulta que **encadene al menos 2** de ellas | 20 |
| 3 | **3 turnos con memoria**, donde el turno 3 dependa del 1 | 20 |
| 4 | **Un límite explícito** que se vea actuando | 20 |
| 5 | Un párrafo (máx. 150 palabras) con **un riesgo** y cómo lo mitigarías | 20 |

**Entrega:** enlace a tu Colab compartido, **con las celdas ya ejecutadas y sus resultados
visibles**. `Compartir → Cualquier persona con el enlace → Lector`.

> ⚠️ **Un notebook sin ejecutar no se puede evaluar.** Las trazas del agente son la
> evidencia de que funciona.

> 💡 **No se pide nada que no hayamos hecho hoy.** Si algo no te sale, entrega igual lo
> que alcanzaste y documenta qué falló: eso puntúa en el requisito 5.
